# EEG preprocessing: raw BioSemi data → epochs

Run this notebook **once per participant**: set `current_participant`, then run the cells in order.
It is interactive: you inspect the raw data to mark bad channels (Step 2) and inspect the ICA components (Step 3), so it cannot run unattended.

Outputs for each participant:
- `epochs/<ID>-epo.fif`: the epochs used by `AcousticCuesEEG_github.ipynb`.
- `logs/<ID>.log`: the final bad-channel list and the ICA components removed.
- `preprocessing/qc_plots/<ID>_evoked_plot.jpg`: a quick check of the average ERP.

Pipeline, as reported in Supplement S3:
1. Re-reference to the average of the two mastoids (EXG1, EXG2).
2. Band-pass filter, 0.1–40 Hz (4th-order zero-phase Butterworth IIR).
3. Mark bad channels by visual inspection, then interpolate them (spherical spline).
4. ICA (FastICA, components explaining 95% of the variance, `random_state=42`). Components correlated with Fp1/Fp2 (used as EOG channels) are found automatically, checked visually, and removed.
5. Remove spurious trigger pulses.
6. Epoch from −100 to 800 ms. Baseline-correct with the mean of −100 to 0 ms. Reject epochs whose peak-to-peak amplitude exceeds 150 µV on any EEG channel.

Requires `experiment_params.py` (channel names, filter settings, paths) and `functions.py` (`filter_short_bursts`) in this folder, and `materials/Reference.csv`.

In [ ]:
# Imports
import ast
import os
from datetime import datetime
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from functions import filter_short_bursts
from experiment_params import params
# Interactive plots need a Qt backend: conda install pyqt

In [ ]:
# Participant and paths (relative to the repository root; this notebook lives in preprocessing/)
current_participant = 105

ROOT         = Path("..")
RAW_DIR      = Path(params["data_dir"])                 # raw .bdf files (not shared)
LOG_DIR      = ROOT / "logs"
BEHAVIOR_DIR = ROOT / "behavior"
REFERENCE    = ROOT / "materials" / "Reference.csv"
EPOCHS_DIR   = ROOT / "epochs"
QC_DIR       = Path("qc_plots")
for d in (LOG_DIR, EPOCHS_DIR, QC_DIR):
    d.mkdir(parents=True, exist_ok=True)

log_path = LOG_DIR / f"{current_participant}.log"

In [ ]:
# Load raw data
raw = mne.io.read_raw_bdf(RAW_DIR / f"{current_participant}.bdf", preload=True)

In [ ]:
# Step 1: Drop unused channels, re-reference to the mastoids, set channel types and montage
raw.drop_channels(params["drop_channels"])
raw.set_channel_types({"EXG1": "misc", "EXG2": "misc"})
raw.set_eeg_reference(["EXG1", "EXG2"])
raw.set_channel_types({"Status": "stim"})  # trigger channel

raw.rename_channels(params["rename_dict"])
raw.set_montage("standard_1020")

# Fp1 and Fp2 serve as EOG channels for ICA, so they are excluded from the EEG channels
raw.set_channel_types({"Fp1": "eog", "Fp2": "eog"})

In [ ]:
%matplotlib qt

# Step 2a: Band-pass filter
raw.filter(l_freq=params["l_freq"], h_freq=params["h_freq"], method="iir", iir_params=params["iir_params"])

# Step 2b: Mark bad channels
# Start from the most recent list in the log (if this participant was run before),
# then inspect the raw data and mark or unmark channels by clicking their names.
lines = log_path.read_text().splitlines(True) if log_path.exists() else []
bad_channels = []
for line in reversed(lines):
    if line.startswith("Bad channels ="):
        bad_channels = ast.literal_eval(line.split("=", 1)[1].split("#")[0].strip().replace("np.str_", ""))
        break
print(f"[{current_participant}] Bad channels from log: {bad_channels}")

raw.info["bads"] = bad_channels
raw.plot(block=True)  # close the window to continue

# Save the final list (replaces any earlier "Bad channels" line)
lines = [l for l in lines if not l.startswith("Bad channels =")]
lines.append(f"Bad channels = {[str(ch) for ch in raw.info['bads']]}  # {datetime.now().isoformat()}\n")
log_path.write_text("".join(lines))
print(f"[{current_participant}] Bad channels logged: {raw.info['bads']}")

In [ ]:
# Step 2c: Interpolate bad channels
raw.interpolate_bads()

In [ ]:
# Step 3a: ICA, with automatic detection of EOG-related components
ica = mne.preprocessing.ICA(n_components=0.95, method="fastica", random_state=42)
ica.fit(raw)

eog_inds, scores = ica.find_bads_eog(raw, ch_name=["Fp1", "Fp2"])
ica.exclude = eog_inds
print(f"[{current_participant}] Automatically detected EOG components: {eog_inds}")

# Inspect: clicking a component in these plots adds or removes it from ica.exclude
ica.plot_components()
ica.plot_sources(raw)

In [ ]:
# Step 3b: Remove the final set of components and log it (run after inspecting the plots above)
print(f"[{current_participant}] Components removed: {ica.exclude}")
raw = ica.apply(raw, exclude=ica.exclude)

lines = [l for l in log_path.read_text().splitlines(True) if not l.startswith("Excluded ICA components:")]
lines.append(f"Excluded ICA components: {[int(i) for i in ica.exclude]}  # {datetime.now().isoformat()}\n")
log_path.write_text("".join(lines))

In [ ]:
# Step 4: Find triggers and remove spurious pulses
all_events = mne.find_events(raw, stim_channel="Status", shortest_event=1)
events = filter_short_bursts(all_events, raw, min_amplitude=20000, min_duration=0.75)
print(f"Triggers found: {len(all_events)} | stimulus onsets kept: {len(events)}")

In [ ]:
# Step 5: Trial information (metadata), one row per stimulus onset
f0_mapping = {1: 5, 3: 3, 5: 1}   # reverses f0 coding for voiceless targets

df = pd.read_csv(BEHAVIOR_DIR / f"{current_participant}_ob.csv")
df = df.dropna(subset=["audio_clip"])                      # drop blank PsychoPy rows
df = df.merge(pd.read_csv(REFERENCE), on="audio_clip", how="left")

df["task-relevant"] = df.apply(lambda r: "yes" if r["target"] in r["Continuum"] else "no", axis=1).astype("category")
df["Continuum"] = df["Continuum"].astype("category")       # beach - peach vs. dear - tear
# Steps from target, coded so that 1 = the target's value
df["vot-stepsfromtarget"] = df.apply(lambda r: r["VOT"] if r["target"] in ["dear", "beach"] else 6 - r["VOT"], axis=1)
df["f0-stepsfromtarget"] = df.apply(lambda r: r["F0"] if r["target"] in ["dear", "beach"] else f0_mapping.get(r["F0"], r["F0"]), axis=1)

# Each trigger must line up with one behavioral trial
assert len(df) == len(events) == 1200, f"{len(df)} trials vs. {len(events)} triggers"

In [ ]:
# Step 6: Epoch, baseline-correct, and reject artifacts
epochs = mne.Epochs(
    raw, events, event_id=None,
    tmin=-0.1, tmax=0.8,
    baseline=(-0.1, 0),                      # subtract the mean of the 100 ms before stimulus onset
    picks=mne.pick_types(raw.info, eeg=True, eog=False, stim=False),
    reject=dict(eeg=150e-6),                 # peak-to-peak amplitude > 150 µV on any EEG channel
    metadata=df, preload=True,
)
print(f"[{current_participant}] Epochs kept: {len(epochs)} of {len(events)}")

In [ ]:
# Save epochs and a quick-check ERP plot
epochs.save(EPOCHS_DIR / f"{current_participant}-epo.fif", overwrite=True)

fig = epochs.average().plot()
fig.savefig(QC_DIR / f"{current_participant}_evoked_plot.jpg")